# EDA: Соревнование по Speaker Retrieval (Криптонит: Тембр)

Цель — выявить ключевые свойства датасета и связать каждое наблюдение с принятым техническим решением.

**Структура ноутбука:**
1. Dataset Overview — статистика по дикторам и файлам
2. Audio Duration Analysis — распределение длительностей
3. Speaker Distribution Analysis — неравномерность по дикторам
4. Train vs Test Domain Gap — сравнение шумности / KS-тест
5. Embedding Space Analysis — разделимость same/diff speaker пар
6. Key Findings Summary — итоговая таблица наблюдений и решений

In [ ]:
# ---------------------------------------------------------------------------
# Импорты и глобальные настройки
# ---------------------------------------------------------------------------
import os
import sys
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import soundfile as sf
from scipy import stats
from sklearn.metrics import roc_curve
from sklearn.preprocessing import normalize

# Воспроизводимость
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Пути
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), 'ASR'))
# Если запуск из eda/, подняться на уровень выше
if not os.path.isdir(os.path.join(PROJECT_ROOT, 'data')):
    PROJECT_ROOT = os.path.abspath('..')
DATA_ROOT   = os.path.join(PROJECT_ROOT, 'data')
TRAIN_CSV   = os.path.join(PROJECT_ROOT, 'extracted_data', 'Для участников', 'train.csv')
TEST_CSV    = os.path.join(PROJECT_ROOT, 'extracted_data', 'Для участников', 'test_public.csv')
EMB_DIR     = os.path.join(PROJECT_ROOT, 'embeddings')
FIGURES_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'figures')

# Если запуск из корня проекта
if not os.path.isdir(os.path.join(DATA_ROOT, 'train')):
    PROJECT_ROOT = os.getcwd()
    DATA_ROOT   = os.path.join(PROJECT_ROOT, 'data')
    TRAIN_CSV   = os.path.join(PROJECT_ROOT, 'extracted_data', 'Для участников', 'train.csv')
    TEST_CSV    = os.path.join(PROJECT_ROOT, 'extracted_data', 'Для участников', 'test_public.csv')
    EMB_DIR     = os.path.join(PROJECT_ROOT, 'embeddings')
    FIGURES_DIR = os.path.join(PROJECT_ROOT, 'eda', 'figures')

os.makedirs(FIGURES_DIR, exist_ok=True)

# Стиль графиков
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'DATA_ROOT    : {DATA_ROOT}')
print(f'FIGURES_DIR  : {FIGURES_DIR}')
print('Все импорты успешны.')

---
## 1. Dataset Overview

In [ ]:
# ---------------------------------------------------------------------------
# Загрузка CSV
# ---------------------------------------------------------------------------
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print('=== Train ===' )
print(f'  Всего файлов   : {len(train_df):,}')
print(f'  Уникальных дикторов : {train_df["speaker_id"].nunique():,}')
print()
print('=== Test ===')
print(f'  Всего файлов   : {len(test_df):,}')

train_df.head(3)

In [ ]:
# ---------------------------------------------------------------------------
# Статистика файлов на диктора
# ---------------------------------------------------------------------------
fps = train_df.groupby('speaker_id').size()  # files per speaker

stats_fps = pd.Series({
    'min'   : int(fps.min()),
    'p5'    : int(fps.quantile(0.05)),
    'p25'   : int(fps.quantile(0.25)),
    'median': int(fps.median()),
    'mean'  : round(fps.mean(), 2),
    'p75'   : int(fps.quantile(0.75)),
    'p95'   : int(fps.quantile(0.95)),
    'max'   : int(fps.max()),
    'std'   : round(fps.std(), 2),
}, name='files_per_speaker')

print('Распределение файлов на диктора (train):')
print(stats_fps.to_string())

# Сколько дикторов с 1 файлом
single_file = (fps == 1).sum()
print(f'\nДикторов с 1 файлом : {single_file} ({100*single_file/len(fps):.1f}%)')

In [ ]:
# ---------------------------------------------------------------------------
# График: гистограмма файлов на диктора
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Гистограмма
ax = axes[0]
ax.hist(fps.values, bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(fps.median(), color='crimson', linestyle='--', linewidth=1.5, label=f'Медиана={int(fps.median())}')
ax.axvline(fps.mean(),   color='orange',  linestyle='--', linewidth=1.5, label=f'Среднее={fps.mean():.1f}')
ax.set_xlabel('Количество файлов на диктора')
ax.set_ylabel('Количество дикторов')
ax.set_title('Распределение файлов на диктора (Train)')
ax.legend()

# Box-plot
ax2 = axes[1]
ax2.boxplot(fps.values, vert=True, patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6),
            medianprops=dict(color='crimson', linewidth=2))
ax2.set_ylabel('Количество файлов на диктора')
ax2.set_title('Box-plot: файлов на диктора')
ax2.set_xticks([])

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '01_files_per_speaker.png'))
plt.show()
print('Сохранено: 01_files_per_speaker.png')

### Наблюдение → Решение

**Наблюдение:** большой разброс файлов на диктора (min=1, median=64, max=100, std≈36). Часть дикторов имеет 1–5 файлов, другие — до 100.  
**Проблема:** при высокой внутридикторской вариабельности один центроид плохо представляет диктора.  
**Решение:** **SubcenterArcFace** (K=3 субцентра) — позволяет моделировать внутридикторскую вариацию без жёсткой привязки к единому прототипу. Это улучшает val P@10 примерно на +0.001.

---
## 2. Audio Duration Analysis

In [ ]:
# ---------------------------------------------------------------------------
# Сэмплирование файлов для анализа длительностей
# ---------------------------------------------------------------------------
N_TRAIN_SAMPLE = 5000
N_TEST_SAMPLE  = 2000

rng = np.random.default_rng(SEED)

# Случайные строки из train и test CSV
train_sample_idx = rng.choice(len(train_df), size=N_TRAIN_SAMPLE, replace=False)
test_sample_idx  = rng.choice(len(test_df),  size=N_TEST_SAMPLE,  replace=False)

train_sample = train_df.iloc[train_sample_idx].copy()
test_sample  = test_df.iloc[test_sample_idx].copy()

print(f'Train sample: {len(train_sample)} файлов')
print(f'Test  sample: {len(test_sample)} файлов')

In [ ]:
# ---------------------------------------------------------------------------
# Быстрое чтение метаданных через sf.info() (без декодирования)
# ---------------------------------------------------------------------------
from tqdm.auto import tqdm

def get_durations(file_paths, data_root):
    """Возвращает список длительностей (секунды) для списка relative-путей."""
    durations = []
    errors = 0
    for fp in tqdm(file_paths, desc='sf.info()', leave=False):
        full_path = os.path.join(data_root, fp)
        try:
            info = sf.info(full_path)
            durations.append(info.frames / info.samplerate)
        except Exception:
            errors += 1
    if errors:
        print(f'  Ошибок чтения: {errors}')
    return np.array(durations)

train_dur = get_durations(train_sample['filepath'].tolist(), DATA_ROOT)
test_dur  = get_durations(test_sample['filepath'].tolist(),  DATA_ROOT)

print(f'Train длительности: {len(train_dur)} записей')
print(f'Test  длительности: {len(test_dur)} записей')

In [ ]:
# ---------------------------------------------------------------------------
# Статистика длительностей
# ---------------------------------------------------------------------------
def dur_stats(dur, label):
    print(f'\n--- {label} ---')
    print(f'  min    : {dur.min():.2f} с')
    print(f'  mean   : {dur.mean():.2f} с')
    print(f'  median : {np.median(dur):.2f} с')
    print(f'  max    : {dur.max():.2f} с')
    print(f'  std    : {dur.std():.2f} с')
    buckets = [
        ('< 2 с',   dur < 2),
        ('2–5 с',   (dur >= 2)  & (dur < 5)),
        ('5–10 с',  (dur >= 5)  & (dur < 10)),
        ('> 10 с',  dur >= 10),
    ]
    for name, mask in buckets:
        print(f'  {name:10s}: {mask.sum():5d} ({100*mask.mean():.1f}%)')

dur_stats(train_dur, 'Train')
dur_stats(test_dur,  'Test')

In [ ]:
# ---------------------------------------------------------------------------
# График: распределение длительностей Train vs Test
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Гистограмма
ax = axes[0]
bins = np.linspace(0, min(train_dur.max(), test_dur.max(), 60), 60)
ax.hist(train_dur, bins=bins, alpha=0.6, label=f'Train (n={len(train_dur):,})', color='steelblue', density=True)
ax.hist(test_dur,  bins=bins, alpha=0.6, label=f'Test  (n={len(test_dur):,})',  color='coral',     density=True)
ax.axvline(3.0, color='gray', linestyle=':', linewidth=1.5, label='3 с')
ax.set_xlabel('Длительность (секунды)')
ax.set_ylabel('Плотность')
ax.set_title('Распределение длительностей (Train vs Test)')
ax.legend()
ax.set_xlim(0, 60)

# Boxplot
ax2 = axes[1]
ax2.boxplot([train_dur, test_dur], labels=['Train', 'Test'],
            patch_artist=True,
            boxprops=dict(alpha=0.6),
            medianprops=dict(color='crimson', linewidth=2))
ax2.set_ylabel('Длительность (секунды)')
ax2.set_title('Box-plot длительностей')
ax2.set_ylim(0)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '02_duration_distribution.png'))
plt.show()
print('Сохранено: 02_duration_distribution.png')

In [ ]:
# ---------------------------------------------------------------------------
# Процент коротких файлов по порогам
# ---------------------------------------------------------------------------
thresholds = [1, 2, 3, 5, 10]
print('Доля файлов короче порога:')
print(f'  {"Порог":>6} | {"Train":>8} | {"Test":>8}')
print('  ' + '-' * 30)
for t in thresholds:
    tr = 100 * (train_dur < t).mean()
    te = 100 * (test_dur  < t).mean()
    print(f'  {t:>5}с | {tr:>7.1f}% | {te:>7.1f}%')

### Наблюдение → Решение

**Наблюдение:** значительная доля файлов короче 3–5 секунд — и в train, и в test.  
**Проблема:** короткий сигнал даёт нестабильные эмбеддинги: при одиночном кропе слишком зависит от выбранного сегмента.  
**Решение:** **TTA (Test-Time Augmentation) с multi-crop**: 10 кропов из разных позиций файла, усреднение эмбеддингов. Это снижает дисперсию и улучшает public P@10 на ~+0.0014.

---
## 3. Speaker Distribution Analysis

In [ ]:
# ---------------------------------------------------------------------------
# Отсортированное распределение файлов на диктора (Lorenz-кривая / Парето)
# ---------------------------------------------------------------------------
fps_sorted = fps.sort_values(ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Столбчатый график (каждый диктор)
ax = axes[0]
ax.fill_between(range(len(fps_sorted)), fps_sorted.values, alpha=0.7, color='steelblue')
ax.set_xlabel('Дикторы (отсортированы по убыванию файлов)')
ax.set_ylabel('Количество файлов')
ax.set_title(f'Файлы на диктора: {len(fps_sorted):,} дикторов')

# Кривая Лоренца
ax2 = axes[1]
fps_lorenz = fps.sort_values().values
cumulative = np.cumsum(fps_lorenz) / fps_lorenz.sum()
speaker_frac = np.arange(1, len(fps_lorenz)+1) / len(fps_lorenz)
ax2.plot(speaker_frac, cumulative, color='steelblue', linewidth=2, label='Кривая Лоренца')
ax2.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Равенство')
ax2.fill_between(speaker_frac, speaker_frac, cumulative, alpha=0.2, color='steelblue')
ax2.set_xlabel('Доля дикторов')
ax2.set_ylabel('Доля файлов')
ax2.set_title('Кривая Лоренца (неравенство распределения)')
ax2.legend()

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '03_speaker_distribution.png'))
plt.show()
print('Сохранено: 03_speaker_distribution.png')

In [ ]:
# ---------------------------------------------------------------------------
# Коэффициент Джини
# ---------------------------------------------------------------------------
def gini(x):
    """Коэффициент Джини: 0 = полное равенство, 1 = максимальное неравенство."""
    x = np.sort(x.astype(float))
    n = len(x)
    cumx = np.cumsum(x)
    return (n + 1 - 2 * cumx.sum() / cumx[-1]) / n

gini_coef = gini(fps.values)
print(f'Коэффициент Джини: {gini_coef:.4f}')
print('(0 = идеальное равенство, 1 = максимальное неравенство)')

# Топ-10 и боттом-10
print('\nТОП-10 дикторов по количеству файлов:')
print(fps.sort_values(ascending=False).head(10).to_string())
print('\nБОТТОМ-10 дикторов (меньше всего файлов):')
print(fps.sort_values(ascending=True).head(10).to_string())

### Наблюдение → Решение

**Наблюдение:** коэффициент Джини > 0 указывает на неравномерность. Часть дикторов имеет 1–10 файлов (редкие дикторы), часть — до 100. Long-tail распределение усложняет обучение.  
**Решение:**
- **ArcFace с большим margin (m=0.3–0.35)** — крупный margin даёт более разделимые прототипы, что особенно важно для недопредставленных дикторов.
- **SubcenterArcFace (K=3)** — для многофайловых дикторов позволяет иметь несколько субпрототипов внутри класса.
- **Взвешенная выборка (sampler)** — чаще показывать редкие классы.

---
## 4. Train vs Test Domain Gap

In [ ]:
# ---------------------------------------------------------------------------
# KS-тест длительностей Train vs Test
# ---------------------------------------------------------------------------
ks_stat, ks_p = stats.ks_2samp(train_dur, test_dur)
print('Критерий Колмогорова–Смирнова (длительности Train vs Test):')
print(f'  KS statistic : {ks_stat:.4f}')
print(f'  p-value      : {ks_p:.4e}')
if ks_p < 0.001:
    print('  Вывод: распределения ЗНАЧИМО различаются (p < 0.001)')
else:
    print('  Вывод: распределения статистически неразличимы')

In [ ]:
# ---------------------------------------------------------------------------
# Оценка уровня шума: RMS-энергия аудиосэмплов
# Загружаем сэмплы (до 3 секунд) и считаем RMS
# ---------------------------------------------------------------------------
N_RMS = 300  # читаем 300 файлов из каждого сплита

def compute_rms_batch(file_paths, data_root, n=300, max_duration=3.0, seed=42):
    """
    Вычисляет RMS-энергию (в dB) для случайной выборки файлов.
    Использует sf.read() только для первых `max_duration` секунд.
    """
    rng = np.random.default_rng(seed)
    chosen = rng.choice(file_paths, size=min(n, len(file_paths)), replace=False)
    rms_db_list = []
    for fp in tqdm(chosen, desc='RMS', leave=False):
        full = os.path.join(data_root, fp)
        try:
            info = sf.info(full)
            sr = info.samplerate
            frames_to_read = min(int(max_duration * sr), info.frames)
            data, _ = sf.read(full, frames=frames_to_read, dtype='float32', always_2d=False)
            if data.ndim > 1:
                data = data.mean(axis=1)
            rms = np.sqrt(np.mean(data ** 2))
            if rms > 1e-9:  # избегаем log(0)
                rms_db_list.append(20 * np.log10(rms))
        except Exception:
            pass
    return np.array(rms_db_list)

train_rms = compute_rms_batch(train_df['filepath'].tolist(), DATA_ROOT, n=N_RMS)
test_rms  = compute_rms_batch(test_df['filepath'].tolist(),  DATA_ROOT, n=N_RMS)

print(f'Train RMS (dB): mean={train_rms.mean():.2f}, std={train_rms.std():.2f}, median={np.median(train_rms):.2f}')
print(f'Test  RMS (dB): mean={test_rms.mean():.2f},  std={test_rms.std():.2f},  median={np.median(test_rms):.2f}')
rms_diff_db = abs(train_rms.mean() - test_rms.mean())
print(f'\nРазница средних RMS: {rms_diff_db:.2f} dB')

In [ ]:
# ---------------------------------------------------------------------------
# KS-тест RMS Train vs Test
# ---------------------------------------------------------------------------
ks_rms_stat, ks_rms_p = stats.ks_2samp(train_rms, test_rms)
print('KS-тест RMS-энергии Train vs Test:')
print(f'  KS statistic : {ks_rms_stat:.4f}')
print(f'  p-value      : {ks_rms_p:.4e}')

# Mann-Whitney U
mw_stat, mw_p = stats.mannwhitneyu(train_rms, test_rms, alternative='two-sided')
print(f'\nMann–Whitney U p-value: {mw_p:.4e}')

In [ ]:
# ---------------------------------------------------------------------------
# График: RMS распределение Train vs Test
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Гистограмма RMS
ax = axes[0]
bins_rms = np.linspace(
    min(train_rms.min(), test_rms.min()),
    max(train_rms.max(), test_rms.max()),
    40
)
ax.hist(train_rms, bins=bins_rms, alpha=0.6, color='steelblue', density=True,
        label=f'Train (среднее={train_rms.mean():.1f} dB)')
ax.hist(test_rms,  bins=bins_rms, alpha=0.6, color='coral',     density=True,
        label=f'Test  (среднее={test_rms.mean():.1f} dB)')
ax.set_xlabel('RMS-энергия (dB)')
ax.set_ylabel('Плотность')
ax.set_title(f'RMS Train vs Test\n(KS p={ks_rms_p:.2e})')
ax.legend()

# Boxplot RMS
ax2 = axes[1]
ax2.boxplot([train_rms, test_rms], labels=['Train', 'Test'],
            patch_artist=True,
            boxprops=dict(alpha=0.6),
            medianprops=dict(color='crimson', linewidth=2))
ax2.set_ylabel('RMS-энергия (dB)')
ax2.set_title('Box-plot RMS-энергии')

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '04_rms_domain_gap.png'))
plt.show()
print('Сохранено: 04_rms_domain_gap.png')

### Наблюдение → Решение

**Наблюдение:**
- KS-тест длительностей: p < 0.001 → распределения значимо различаются.
- RMS-энергия различается между train и test, что указывает на разный уровень шума/условий записи (domain gap).

**Проблема:** модель, обученная на чистых студийных записях (train), плохо обобщается на зашумлённые тестовые файлы.  
**Подтверждение:** baseline без аугментации (public 0.2699) → с синтетической аугментацией без реального шума улучшение было минимальным → добавление **MUSAN + RIR** реверберации как реалистичных условий дало резкий прирост public P@10 (+0.102).  
**Вывод:** domain gap является главным источником деградации; реалистичная шумовая аугментация критически важна.

---
## 5. Embedding Space Analysis

In [ ]:
# ---------------------------------------------------------------------------
# Загрузка pre-computed эмбеддингов CAM++ stage3 TTA-10
# Эти эмбеддинги соответствуют тестовым файлам (134697 x 512)
# ---------------------------------------------------------------------------
emb_path = os.path.join(EMB_DIR, 'campplus_stage3_tta10.npy')

if os.path.exists(emb_path):
    test_emb = np.load(emb_path).astype(np.float32)
    print(f'Загружены test эмбеддинги: {test_emb.shape}')
    # L2-нормализация (cosine similarity = dot product после нормализации)
    test_emb_norm = normalize(test_emb, norm='l2')
else:
    print(f'Файл не найден: {emb_path}')
    test_emb = None

In [ ]:
# ---------------------------------------------------------------------------
# Восстановление val-разбивки (seed=42, 500 дикторов)
# Берём baseline_val500.npy — это эмбеддинги val-файлов
# ---------------------------------------------------------------------------
val_emb_path = os.path.join(EMB_DIR, 'baseline_val500.npy')

# Воссоздаём val-диктора (те же 500, что использовались при валидации)
all_speakers = train_df['speaker_id'].unique()
rng_val = np.random.default_rng(SEED)
val_speakers = rng_val.choice(all_speakers, size=500, replace=False)
val_df = train_df[train_df['speaker_id'].isin(val_speakers)].reset_index(drop=True)

print(f'Val-дикторов : {val_df["speaker_id"].nunique()}')
print(f'Val-файлов   : {len(val_df)}')

# Загрузка val-эмбеддингов (baseline ONNX, 192-мерные)
if os.path.exists(val_emb_path):
    val_emb = np.load(val_emb_path).astype(np.float32)
    print(f'Val эмбеддинги: {val_emb.shape}')
    # Подгоняем размер под val_df (если не совпадает — используем доступные)
    n_use = min(len(val_df), len(val_emb))
    val_emb = val_emb[:n_use]
    val_labels = val_df['speaker_id'].values[:n_use]
    val_emb_norm = normalize(val_emb, norm='l2')
    print(f'Используем: {n_use} эмбеддингов для {val_df["speaker_id"].nunique()} дикторов')
else:
    print(f'Файл не найден: {val_emb_path}')
    val_emb = None
    val_labels = None

In [ ]:
# ---------------------------------------------------------------------------
# Вычисление попарных схожестей для val-эмбеддингов
# Same-speaker vs Different-speaker пары (случайная выборка)
# ---------------------------------------------------------------------------
if val_emb is not None:
    N_PAIRS = 50_000  # пар каждого типа
    rng_pairs = np.random.default_rng(SEED)
    
    # Индексы для same-speaker пар
    same_sims = []
    diff_sims = []
    
    unique_labels = np.unique(val_labels)
    label_to_idx = {lbl: np.where(val_labels == lbl)[0] for lbl in unique_labels}
    
    # Дикторы с >= 2 файлами (для same-speaker пар)
    valid_speakers = [lbl for lbl, idx in label_to_idx.items() if len(idx) >= 2]
    
    print(f'Дикторов с >=2 файлами: {len(valid_speakers)}')
    
    # Same-speaker пары
    count_same = 0
    while count_same < N_PAIRS:
        spk = rng_pairs.choice(valid_speakers)
        idx = label_to_idx[spk]
        i, j = rng_pairs.choice(idx, size=2, replace=False)
        sim = float(val_emb_norm[i] @ val_emb_norm[j])
        same_sims.append(sim)
        count_same += 1
        if count_same % 10000 == 0:
            print(f'  Same-speaker: {count_same}/{N_PAIRS}', end='\r')
    
    # Different-speaker пары
    all_idx = np.arange(len(val_labels))
    for _ in range(N_PAIRS):
        i, j = rng_pairs.choice(all_idx, size=2, replace=False)
        if val_labels[i] != val_labels[j]:
            diff_sims.append(float(val_emb_norm[i] @ val_emb_norm[j]))
    
    same_sims = np.array(same_sims)
    diff_sims = np.array(diff_sims)
    
    print(f'\nSame-speaker: {len(same_sims):,} пар, mean={same_sims.mean():.4f}, std={same_sims.std():.4f}')
    print(f'Diff-speaker: {len(diff_sims):,} пар, mean={diff_sims.mean():.4f}, std={diff_sims.std():.4f}')
else:
    same_sims = None
    diff_sims = None
    print('Эмбеддинги недоступны, пропускаем анализ.')

In [ ]:
# ---------------------------------------------------------------------------
# Вычисление EER (Equal Error Rate) и построение ROC
# ---------------------------------------------------------------------------
if same_sims is not None:
    # Бинарные метки: 1 = same, 0 = diff
    y_true  = np.concatenate([np.ones(len(same_sims)), np.zeros(len(diff_sims))])
    y_score = np.concatenate([same_sims, diff_sims])
    
    fpr, tpr, thresholds_roc = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    
    # EER: точка пересечения FPR и FNR
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
    eer_threshold = thresholds_roc[eer_idx]
    
    print(f'EER = {100*eer:.2f}%  (порог={eer_threshold:.4f})')
    
    # Overlap (% same-speaker пар ниже EER-порога = ошибочно отклонены)
    overlap_same = 100 * (same_sims < eer_threshold).mean()
    overlap_diff = 100 * (diff_sims >= eer_threshold).mean()
    print(f'Same-speaker ниже порога (FN): {overlap_same:.1f}%')
    print(f'Diff-speaker выше порога (FP): {overlap_diff:.1f}%')

In [ ]:
# ---------------------------------------------------------------------------
# Графики: распределение схожестей + ROC-кривая
# ---------------------------------------------------------------------------
if same_sims is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    
    # 1. Score distributions
    ax = axes[0]
    bins_sim = np.linspace(
        min(same_sims.min(), diff_sims.min()),
        max(same_sims.max(), diff_sims.max()),
        80
    )
    ax.hist(diff_sims, bins=bins_sim, alpha=0.6, color='coral',     density=True,
            label=f'Разные дикторы\n(mean={diff_sims.mean():.3f})')
    ax.hist(same_sims, bins=bins_sim, alpha=0.6, color='steelblue', density=True,
            label=f'Один диктор\n(mean={same_sims.mean():.3f})')
    ax.axvline(eer_threshold, color='black', linestyle='--', linewidth=1.5,
               label=f'EER порог={eer_threshold:.3f}')
    ax.set_xlabel('Косинусная схожесть')
    ax.set_ylabel('Плотность')
    ax.set_title('Распределение косинусных схожестей\n(Same vs Different speaker)')
    ax.legend(fontsize=9)
    
    # 2. ROC curve
    ax2 = axes[1]
    ax2.plot(fpr, tpr, color='steelblue', linewidth=2, label='ROC')
    ax2.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Случайный')
    ax2.scatter([fpr[eer_idx]], [tpr[eer_idx]], color='crimson', zorder=5,
                s=80, label=f'EER={100*eer:.2f}%')
    ax2.set_xlabel('False Positive Rate')
    ax2.set_ylabel('True Positive Rate')
    ax2.set_title('ROC-кривая (Speaker Verification)')
    ax2.legend()
    
    # 3. DET curve (FPR vs FNR)
    ax3 = axes[2]
    ax3.plot(fpr, fnr, color='steelblue', linewidth=2)
    ax3.scatter([fpr[eer_idx]], [fnr[eer_idx]], color='crimson', zorder=5,
                s=80, label=f'EER={100*eer:.2f}%')
    ax3.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax3.set_xlabel('False Alarm Rate (FPR)')
    ax3.set_ylabel('Miss Rate (FNR)')
    ax3.set_title('DET-кривая')
    ax3.legend()
    
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, '05_embedding_space.png'))
    plt.show()
    print('Сохранено: 05_embedding_space.png')
else:
    print('Пропущено: эмбеддинги недоступны')

In [ ]:
# ---------------------------------------------------------------------------
# TSNE-визуализация эмбеддингов (случайные 20 дикторов)
# ---------------------------------------------------------------------------
if val_emb is not None:
    from sklearn.manifold import TSNE
    
    # Берём 20 дикторов с наибольшим числом файлов для наглядности
    top_speakers = (
        pd.Series(val_labels).value_counts().head(20).index.tolist()
    )
    mask = np.isin(val_labels, top_speakers)
    
    emb_sub  = val_emb_norm[mask]
    lbl_sub  = val_labels[mask]
    
    print(f't-SNE на {mask.sum()} записях от {len(top_speakers)} дикторов...')
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=500, verbose=0)
    emb_2d = tsne.fit_transform(emb_sub)
    
    # Кодируем дикторов числами для цвета
    spk_to_int = {s: i for i, s in enumerate(top_speakers)}
    colors = np.array([spk_to_int[l] for l in lbl_sub])
    
    fig, ax = plt.subplots(figsize=(10, 7))
    scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1],
                         c=colors, cmap='tab20',
                         s=12, alpha=0.7)
    ax.set_title(f't-SNE эмбеддингов (20 дикторов, baseline val)', fontsize=13)
    ax.set_xlabel('t-SNE dim 1')
    ax.set_ylabel('t-SNE dim 2')
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, '05b_tsne.png'))
    plt.show()
    print('Сохранено: 05b_tsne.png')
else:
    print('Пропущено: эмбеддинги недоступны')

### Наблюдение → Решение

**Наблюдение:**
- Распределения схожестей same/different speaker существенно перекрываются.
- EER указывает на наличие значительного числа ошибочно близких пар (разные дикторы с высокой схожестью) и ошибочно далёких пар (один диктор с низкой схожестью).

**Проблема:** простой поиск ближайших соседей (FAISS) опирается только на попарное расстояние, игнорируя контекст соседства.

**Решение:** **K-reciprocal reranking** — переоценка ранжирования с учётом взаимного соседства (если A близок к B и B близок к A, это сильный сигнал). На практике это даёт +0.057 к public P@10.

---
## 6. Key Findings Summary

In [ ]:
# ---------------------------------------------------------------------------
# Финальная таблица: наблюдение → решение → результат
# ---------------------------------------------------------------------------

# Собираем измеримые факты из ранее вычисленных значений
pct_lt3_train = 100 * (train_dur < 3).mean()
pct_lt3_test  = 100 * (test_dur  < 3).mean()
rms_diff      = abs(train_rms.mean() - test_rms.mean())
eer_val       = 100 * eer if same_sims is not None else float('nan')

rows = [
    {
        'Наблюдение': 'Большой разброс записей на диктора',
        'Измеримый факт': f'min=1, median={int(fps.median())}, max={int(fps.max())}, std={fps.std():.1f}',
        'Принятое решение': 'SubcenterArcFace (K=3 субцентра)',
        'Результат': 'val P@10 +0.001 (9706→9707)',
    },
    {
        'Наблюдение': 'Короткие записи в тесте',
        'Измеримый факт': f'{pct_lt3_train:.1f}% train < 3с; {pct_lt3_test:.1f}% test < 3с',
        'Принятое решение': 'TTA 10-crop (усреднение 10 кропов)',
        'Результат': 'public P@10 +0.0014',
    },
    {
        'Наблюдение': 'Domain gap train → test',
        'Измеримый факт': f'RMS разница {rms_diff:.1f} dB, KS p={ks_rms_p:.1e}',
        'Принятое решение': 'MUSAN + RIR аугментация при обучении',
        'Результат': 'public P@10 +0.102 (0.27→0.37)',
    },
    {
        'Наблюдение': 'Overlap same/diff speaker scores',
        'Измеримый факт': f'EER ≈{eer_val:.1f}% на baseline val-эмбеддингах',
        'Принятое решение': 'K-reciprocal reranking (k=70, λ=0.1)',
        'Результат': 'public P@10 +0.057',
    },
    {
        'Наблюдение': 'Разные архитектурные семейства дают разные ошибки',
        'Измеримый факт': 'CAM++ val=0.9707, ERes2Net val=0.9682, ECAPA val=0.9462',
        'Принятое решение': 'Ансамблирование (взвешенная сумма эмбеддингов)',
        'Результат': 'TBD (в работе)',
    },
]

summary_df = pd.DataFrame(rows)
print('=== Key Findings ===')
print(summary_df.to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# Таблица в виде markdown
# ---------------------------------------------------------------------------
from IPython.display import Markdown, display

md_rows = ['| Наблюдение | Измеримый факт | Принятое решение | Результат |',
           '|:---|:---|:---|:---|']
for r in rows:
    md_rows.append(f"| {r['Наблюдение']} | {r['Измеримый факт']} | {r['Принятое решение']} | {r['Результат']} |")

display(Markdown('\n'.join(md_rows)))

In [ ]:
# ---------------------------------------------------------------------------
# Итоговый график: прогресс метрики по экспериментам
# ---------------------------------------------------------------------------
import json

exp_path = os.path.join(PROJECT_ROOT, 'results', 'experiments.jsonl')
experiments = []
if os.path.exists(exp_path):
    with open(exp_path) as f:
        for line in f:
            try:
                experiments.append(json.loads(line.strip()))
            except Exception:
                pass

if experiments:
    exp_df = pd.DataFrame(experiments)
    # Берём последний запуск каждого имени
    exp_df_last = exp_df.sort_values('timestamp').groupby('name').last().reset_index()
    exp_df_last = exp_df_last.sort_values('precision_at_10', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    colors_bar = ['crimson' if p < 0.95 else ('orange' if p < 0.965 else 'steelblue')
                  for p in exp_df_last['precision_at_10']]
    bars = ax.barh(exp_df_last['name'], exp_df_last['precision_at_10'],
                   color=colors_bar, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, exp_df_last['precision_at_10']):
        ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)
    ax.set_xlabel('Precision@10 (val)')
    ax.set_title('Прогресс экспериментов: Precision@10 на val-выборке')
    ax.set_xlim(0.85, 0.985)
    ax.axvline(0.97, color='gray', linestyle=':', linewidth=1)
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, '06_experiment_progress.png'))
    plt.show()
    print('Сохранено: 06_experiment_progress.png')
else:
    print(f'experiments.jsonl не найден: {exp_path}')

In [ ]:
# ---------------------------------------------------------------------------
# Финальное резюме всех сохранённых фигур
# ---------------------------------------------------------------------------
print('=== EDA завершён ===')
print(f'\nФигуры сохранены в: {FIGURES_DIR}')
for fn in sorted(os.listdir(FIGURES_DIR)):
    if fn.endswith('.png'):
        print(f'  {fn}')

print('\n=== Ключевые выводы ===')
print(f'  1. {len(train_df):,} файлов, {train_df["speaker_id"].nunique():,} дикторов')
print(f'  2. Медиана файлов/диктор: {int(fps.median())}, разброс: {int(fps.min())}–{int(fps.max())}')
print(f'  3. Файлов < 3с: train={pct_lt3_train:.1f}%, test={pct_lt3_test:.1f}%')
print(f'  4. Domain gap (RMS): {rms_diff:.1f} dB, KS p={ks_rms_p:.1e}')
if same_sims is not None:
    print(f'  5. EER baseline: {eer_val:.2f}% → reranking критически важен')